# Reprise complète — fréquence × gravité avec XGBoost

L'objectif est la RMSE de prime, avec deux composantes distinctes. Le notebook original est conservé. Les cellules suivantes lisent les résultats reproductibles de `scripts/restart_xgboost.py all` ; elles ne relancent pas la recherche coûteuse. Les associations descriptives ne sont pas des effets causaux.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p/'run.py').exists())
R = ROOT/'reports/restart_xgboost'
sys.path.insert(0,str(ROOT/'src'))
from pricing.experiment import load_data
train,test,_,_=load_data()
audit=json.loads((R/'data_audit.json').read_text())
audit

{'train_rows': 50000,
 'test_rows': 50000,
 'clients': 45785,
 'claims': 2917,
 'anomalies': {'zero_weight': 1883,
  'zero_engine': 3,
  'licence_exceeds_age1': 12,
  'licence_exceeds_age2': 397,
  'largest_25_share_squared_cost': 0.21505416650805687},
 'note': 'Associations are descriptive, not causal. Duration/exposure and sale-date semantics unconfirmed.'}

## Dictionnaire et qualité des variables

Les unités non documentées restent signalées. Les identifiants et les sexes sont exclus. Les zéros du second conducteur représentent son absence. Poids/cylindrée nuls et permis plus anciens que le conducteur sont comparés sous deux traitements : valeurs brutes avec indicateurs, ou valeurs manquantes avec indicateurs. Aucune suppression des sinistres extrêmes.

In [2]:
dictionary=pd.read_csv(R/'variable_dictionary.csv')
pd.set_option('display.max_rows',40)
dictionary

,variable,meaning,role,train_missing,train_unique,test_missing,test_unseen,minimum,maximum,train_test_KS
0,index,Submission row identifier,excluded / target / grouping,0,50000,0.0,NaN,0.0,49999.00,1.000000
1,id_client,Client grouping identifier,excluded / target / grouping,0,45785,NaN,NaN,NaN,NaN,NaN
2,id_vehicule,Vehicle identifier,excluded / target / grouping,0,5,NaN,NaN,NaN,NaN,NaN
3,id_contrat,Contract identifier,excluded / target / grouping,0,50000,NaN,NaN,NaN,NaN,NaN
4,bonus,Bonus-malus coefficient,predictor,0,73,0.0,NaN,0.5,2.16,0.005340
5,type_contrat,Coverage level,predictor,0,4,0.0,0.0,NaN,NaN,NaN
6,duree_contrat,Contract duration; unit unconfirmed,predictor,0,41,0.0,NaN,1.0,41.00,0.002980
7,anciennete_info,Information tenure; unit unconfirmed,predictor,0,25,0.0,NaN,1.0,25.00,0.003020
8,freq_paiement,Payment frequency,predictor,0,4,0.0,0.0,NaN,NaN,NaN
9,paiement,Payment method,predictor,0,2,0.0,0.0,NaN,NaN,NaN


In [3]:
groups=pd.read_csv(R/'variable_groups.csv')
contract=groups[groups.variable.eq('type_contrat')].set_index('level')
fig,ax=plt.subplots(1,2,figsize=(11,4))
contract.claim_rate.plot.bar(ax=ax[0],title='Probabilité descriptive par contrat')
contract.conditional_severity.plot.bar(ax=ax[1],title='Coût moyen parmi les sinistres')
plt.tight_layout();plt.show()
contract

C:\Users\fourr\AppData\Local\Temp\ipykernel_2968\1413947017.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.show()


,variable,rows,claims,claim_rate,mean_cost,conditional_severity
level,,,,,,
Maxi,type_contrat,32360,2421,0.074815,138.036290,1845.045171
Median1,type_contrat,4638,139,0.029970,45.350220,1513.196547
Median2,type_contrat,8759,283,0.032310,44.162714,1366.859399
Mini,type_contrat,4243,74,0.017440,24.618739,1411.585270


In [4]:
claims=train.loc[train.nombre_sinistres.eq(1),'montant_sinistre']
fig,ax=plt.subplots(figsize=(8,4))
ax.hist(claims,bins=np.geomspace(claims.min(),claims.max(),45));ax.set_xscale('log')
ax.set(xlabel='Montant (échelle logarithmique)',ylabel='Effectif',title='Gravité : forte asymétrie et grandes valeurs conservées')
plt.show()
claims.describe(percentiles=[.5,.9,.99])

C:\Users\fourr\AppData\Local\Temp\ipykernel_2968\325772892.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


count     2917.000000
mean      1771.843401
std       1463.954673
min        600.600000
50%       1236.000000
90%       3336.740000
99%       6963.118800
max      21826.960000
Name: montant_sinistre, dtype: float64

## Protocole

Trois plis externes séparés par client évaluent toute la sélection. Chacun contient cinq plis internes, 60 configurations de fréquence et 60 de gravité. Chaque ajustement choisit son nombre d'arbres sur un sous-ensemble de clients de son apprentissage, puis réapprend avec ce nombre d'arbres. Les catégories sont apprises localement.

Fréquence : `binary:logistic`, sans repondération des classes. Gravité : `reg:gamma` ou `reg:squarederror`, uniquement sur les sinistres positifs. La calibration logistique est croisée sur les prédictions internes hors pli ; son éventuel optimisme est inclus dans l'évaluation externe. Les cinq meilleures configurations de chaque composante sont comparées, puis leurs mélanges avec la référence sont sélectionnés sur la RMSE de prime.

Les données ayant déjà été explorées, ces plis ne constituent pas un nouveau jeu indépendant. La recherche sur l'ensemble du train ne mesure pas la généralisation.

In [5]:
results=json.loads((R/'nested_results.json').read_text())
pd.DataFrame({'Sélection XGBoost':results['selected'],'Référence initiale':results['reference']})

,Sélection XGBoost,Référence initiale
MSE,293902.846967,293829.184497
RMSE,542.128072,542.060130
MAE,191.011443,190.861365
bias,-0.458240,-0.329742
predicted_observed_ratio,0.995567,0.996810


In [6]:
pd.DataFrame([{'pli':z['fold'],'RMSE sélection':z['selected']['RMSE'],'RMSE référence':z['reference']['RMSE']} for z in results['folds']])

,pli,RMSE sélection,RMSE référence
0,1,569.608847,569.562715
1,2,518.052611,518.131010
2,3,537.471388,537.239063


In [7]:
print('Gain RMSE et intervalle bootstrap clients :',results['gain'])
print('Fréquence :',results['frequency'])
print('Gravité conditionnelle :',results['severity'])
pd.read_csv(R/'contract_performance.csv')

Gain RMSE et intervalle bootstrap clients : {'gain_RMSE': -0.0679425117447181, 'interval_95': [-0.21500735120780518, 0.0653531128292056], 'repetitions': 2000, 'interpretation': 'Descriptive paired client bootstrap conditional on fitted predictions; not full training uncertainty.'}
Fréquence : {'log_loss': 0.21398844626168298, 'Brier': 0.05407264677374027, 'AUC': 0.6525722585715795}
Gravité conditionnelle : {'MSE': 2116317.3216845198, 'RMSE': 1454.7567912488053, 'MAE': 933.8615113544114, 'bias': -7.287213476005125, 'predicted_observed_ratio': 0.9958872135805552}


,type_contrat,n,MSE,RMSE,MAE,bias,predicted_observed_ratio,reference_RMSE
0,Maxi,32360,393331.014542,627.161075,249.679232,-4.485633,0.967504,627.095262
1,Median1,4638,194562.758677,441.092687,90.419295,3.053240,1.067326,440.933246
2,Median2,8759,84883.235101,291.347276,93.252332,8.725678,1.197580,291.295031
3,Mini,4243,75672.021391,275.085480,55.335930,7.460268,1.303032,275.045630


In [8]:
cal=pd.read_csv(R/'calibration.csv')
fig,ax=plt.subplots(figsize=(5,5));ax.plot(cal.predicted,cal.observed,'o-')
upper=max(cal.predicted.max(),cal.observed.max());ax.plot([0,upper],[0,upper],'--',color='gray')
ax.set(xlabel='Probabilité prédite',ylabel='Taux observé',title='Calibration sur les plis externes');plt.show()
cal

C:\Users\fourr\AppData\Local\Temp\ipykernel_2968\1850297755.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(xlabel='Probabilité prédite',ylabel='Taux observé',title='Calibration sur les plis externes');plt.show()


,bin,n,predicted,observed
0,"(0.0015799999999999998, 0.0214]",5000,0.014897,0.0134
1,"(0.0214, 0.0293]",5000,0.025397,0.0270
2,"(0.0293, 0.0379]",5000,0.033493,0.0310
3,"(0.0379, 0.0495]",5000,0.043468,0.0404
4,"(0.0495, 0.06]",5000,0.055044,0.0600
5,"(0.06, 0.0671]",5000,0.063644,0.0610
6,"(0.0671, 0.0742]",5000,0.070651,0.0708
7,"(0.0742, 0.0825]",5000,0.078110,0.0778
8,"(0.0825, 0.0949]",5000,0.088158,0.0928
9,"(0.0949, 0.336]",5000,0.110514,0.1092


## Sélection finale et fichier Kaggle

La recherche finale est ajustée sur tous les clients après l'évaluation externe. Si l'intervalle du gain inclut zéro, la référence reste recommandée pour la production ; le nouveau fichier est identifié comme challenger. Un score Kaggle inférieur à 1365.500 doit être constaté après soumission et ne se déduit pas du score local.

In [9]:
final=json.loads((R/'final_model.json').read_text())
print('Choix final :',final['choice'])
print('Recommandation :',final['production_recommendation'])
print('Score Kaggle :',final['kaggle_score'])
submission=pd.read_csv(ROOT/final['challenger'])
assert list(submission)==['index','pred'] and len(submission)==50000
assert np.array_equal(submission['index'],test['index'])
assert np.isfinite(submission.pred).all() and submission.pred.ge(0).all()
submission.head()

Choix final : {'frequency': {'kind': 'xgb', 'trial': 21, 'calibrated': True, 'weight': 0.5}, 'severity': {'kind': 'xgb', 'trial': 48, 'weight': 0.5}, 'selection_RMSE': 541.9003006389393, 'reference_RMSE': 541.981085620689}
Recommandation : original reference
Score Kaggle : 1366.025


,index,pred
0,50000,45.325648
1,50001,40.694918
2,50002,171.098286
3,50003,125.199406
4,50004,29.679914
